# ATLAS hydro CMIP6 scenarios download

This notebook downloads CMIP6 scenario data from an ESGF search node.
Documentation: https://esgf.github.io/esgf-user-support/metagrid.html

The workflow is organised step by step so that it can be reused by people who are not expert Python users.  
The default example downloads river discharge (`rivo`) from CMIP6 for one model and one or more experiments. Documentation: https://cmip6dr.github.io/Data_Request_Home/

The notebook does not use absolute paths. All downloaded files are saved inside the project folder, under:

`../data/cmip6/downloads/{variable}/global/{experiment}/{model}/`


## Step 1. Import libraries

This step loads the Python libraries needed to search CMIP6 datasets on ESGF and download NetCDF files.

If `pyesgf` is not installed in your environment, install it before running the notebook, for example with:

```bash
pip install pyesgf
```


In [1]:
from pathlib import Path
import re
import sys

import requests
from pyesgf.search import SearchConnection

## Step 2. Define user parameters

Edit only this cell to change model, variable, experiment, or output paths.

For CMIP6 data, the download is global by default. Country level clipping or basin level aggregation should be done in the following preprocessing notebooks.

Parameter meaning:

- `country`: country name used only to organise the output folders. The ESGF download itself is not clipped by country.
- `model`: CMIP6 model name.
- `experiments`: list of CMIP6 experiments to download. For example `historical`, `ssp245`, `ssp370`, `ssp585`. Documentation: https://www.ipcc.ch/report/emissions-scenarios/
- `variable`: CMIP6 variable name to download. Documentation: https://cmip6dr.github.io/Data_Request_Home/
- `table`: CMIP6 table identifier, for example `Eday`.
- `variant_preference`: preferred CMIP6 ensemble members. The notebook tries them in order and uses the first available one.
- `search_node`: ESGF search node.
- `BASE_OUTPUT_DIR`: root folder where all downloaded ESGF files will be stored.

In [2]:
# ---------------------------------------------------------------------
# Main configuration
# ---------------------------------------------------------------------

# ESGF search node.
# DKRZ is used here because it is often stable for CMIP6 searches.
SEARCH_NODE = "https://esgf-data.dkrz.de/esg-search"

# CMIP6 model to download.
MODEL = "CNRM-ESM2-1"

# CMIP6 variable.
# For hydrology, 'rivo' is river discharge.
VARIABLE = "rivo"

# CMIP6 table.
# For daily river discharge, use 'Eday'.
TABLE = "Eday"

# Experiments to download.
# Common options are:
# 'historical', 'ssp370', 'ssp585'
EXPERIMENTS = ["historical",
               'ssp370',
               'ssp585',
              ]

# Preferred realizations / variants.
# The notebook will try these variants in order and use the first available one.
VARIANT_PREF = [
    "r1i1p1f1",
    "r1i1p1f2",
]

# Generic output folder.
# Files will be saved in:
# ../data/cmip6/downloads/{VARIABLE}/global/{experiment}/{MODEL}/
BASE_OUTPUT_DIR = Path("../data/esgf_downloads")

# Download chunk size.
CHUNK_SIZE = 1 << 20  # 1 MiB

# If True, existing files are not downloaded again.
SKIP_EXISTING_FILES = True

## Step 3. Define time ranges

CMIP6 historical simulations usually end in 2014. Scenario simulations usually start in 2015.

The function below automatically assigns the correct year range depending on the experiment.


In [3]:
def get_year_range_for_experiment(experiment):
    """
    Return the default download period for a CMIP6 experiment.

    Parameters
    ----------
    experiment : str
        CMIP6 experiment ID, for example 'historical' or 'ssp370'.

    Returns
    -------
    tuple
        Start year and end year.
    """

    if experiment == "historical":
        return 1950, 2014

    return 2015, 2100

## Step 4. Helper functions

These functions keep the main workflow short and readable.

They do four things:

1. extract the first year from each CMIP6 file name
2. safely download a file
3. search for the first available model variant
4. select only files inside the requested year range


In [4]:
def extract_start_year_from_filename(filename):
    """
    Extract the start year from a standard CMIP6 file name.

    CMIP6 files usually contain a time range like:
    ..._YYYYMMDD-YYYYMMDD.nc

    Example:
    rivo_Eday_CNRM-ESM2-1_ssp370_r1i1p1f2_gr_20150101-20151231.nc
    """

    match = re.search(r"_(\d{4})\d{4}-", filename)

    if match:
        return int(match.group(1))

    return None


def fetch_file(url, destination, chunk_size=CHUNK_SIZE, skip_existing=True):
    """
    Download one file from a URL.

    Existing files are skipped by default.
    """

    if destination.exists() and skip_existing:
        print(f"   File already exists, skipping: {destination.name}")
        return

    print(f"   Downloading: {destination.name}")

    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()

        with destination.open("wb") as output_file:
            for chunk in response.iter_content(chunk_size):
                if chunk:
                    output_file.write(chunk)


def find_first_available_dataset(connection, experiment):
    """
    Search the ESGF catalogue and return the first dataset matching the
    preferred variants listed in VARIANT_PREF.
    """

    for variant in VARIANT_PREF:
        context = connection.new_context(
            project="CMIP6",
            source_id=MODEL,
            experiment_id=experiment,
            variable_id=VARIABLE,
            table_id=TABLE,
            variant_label=variant,
            latest=True,
            facets="project,experiment_id,source_id,variant_label",
        )

        results = context.search()

        if results:
            selected_dataset = results[0]
            print(f"   Selected variant: {variant}")
            print(f"   Dataset: {selected_dataset.dataset_id}")
            return selected_dataset

    return None


def filter_files_by_year(dataset, year_start, year_end):
    """
    Select NetCDF files whose start year is inside the requested period.
    """

    selected_files = []

    for file_result in dataset.file_context().search():
        if not file_result.filename.endswith(".nc"):
            continue

        start_year = extract_start_year_from_filename(file_result.filename)

        if start_year is None:
            continue

        if year_start <= start_year <= year_end:
            selected_files.append(file_result)

    return selected_files

## Step 5. Connect to ESGF

This cell opens the connection to the selected ESGF search node.

`distrib=False` searches only the selected node. This is usually easier to debug.  
If you need a broader search across ESGF, you can try `distrib=True`.


In [5]:
connection = SearchConnection(
    SEARCH_NODE,
    distrib=False,
)

print("Connected to ESGF search node:")
print(SEARCH_NODE)

Connected to ESGF search node:
https://esgf-data.dkrz.de/esg-search


## Step 6. Download CMIP6 files

This is the main workflow.

For each experiment, the notebook:

1. defines the correct year range
2. searches for the first available variant
3. creates a clean output folder
4. filters the available NetCDF files by year
5. downloads the selected files


In [6]:
for experiment in EXPERIMENTS:
    year_start, year_end = get_year_range_for_experiment(experiment)

    print("\n" + "=" * 70)
    print(f"Experiment: {experiment}")
    print(f"Years: {year_start} to {year_end}")
    print("=" * 70)

    selected_dataset = find_first_available_dataset(
        connection=connection,
        experiment=experiment,
    )

    if selected_dataset is None:
        print("   No matching dataset found. Skipping this experiment.")
        continue

    output_dir = (
        BASE_OUTPUT_DIR
        / MODEL
        / experiment
        / VARIABLE

    )

    output_dir.mkdir(parents=True, exist_ok=True)

    files_to_download = filter_files_by_year(
        dataset=selected_dataset,
        year_start=year_start,
        year_end=year_end,
    )

    if not files_to_download:
        print("   No NetCDF files found in the selected time range.")
        continue

    print(f"   Files selected: {len(files_to_download)}")
    print(f"   Output folder: {output_dir}")

    for file_result in files_to_download:
        destination = output_dir / file_result.filename

        try:
            fetch_file(
                url=file_result.download_url,
                destination=destination,
                chunk_size=CHUNK_SIZE,
                skip_existing=SKIP_EXISTING_FILES,
            )

        except Exception as error:
            print(f"   Error while downloading {file_result.filename}")
            print(f"   {error}")
            sys.exit(1)

    print(f"   Completed experiment: {experiment}")
    print(f"   Files saved in: {output_dir}")


Experiment: historical
Years: 1950 to 2014
   Selected variant: r1i1p1f2
   Dataset: CMIP6.CMIP.CNRM-CERFACS.CNRM-ESM2-1.historical.r1i1p1f2.Eday.rivo.gn.v20181206|esgf3.dkrz.de
   Files selected: 2
   Output folder: ../data/esgf_downloads/CNRM-ESM2-1/historical/rivo
   Downloading: rivo_Eday_CNRM-ESM2-1_historical_r1i1p1f2_gn_19500101-19991231.nc
   Downloading: rivo_Eday_CNRM-ESM2-1_historical_r1i1p1f2_gn_20000101-20141231.nc
   Completed experiment: historical
   Files saved in: ../data/esgf_downloads/CNRM-ESM2-1/historical/rivo


## Step 7. Check downloaded files

After the download, this cell lists the NetCDF files saved in the output folders.

This is a quick check to verify that the download worked correctly.


In [7]:
for experiment in EXPERIMENTS:
    output_dir = (
        BASE_OUTPUT_DIR
        / VARIABLE
        / "global"
        / experiment
        / MODEL
    )

    downloaded_files = sorted(output_dir.glob("*.nc"))

    print("\n" + "=" * 70)
    print(f"Experiment: {experiment}")
    print(f"Output folder: {output_dir}")
    print(f"Number of downloaded files: {len(downloaded_files)}")

    for file_path in downloaded_files[:10]:
        print(f"   {file_path.name}")

    if len(downloaded_files) > 10:
        print(f"   ... and {len(downloaded_files) - 10} more files")


Experiment: historical
Output folder: ../data/esgf_downloads/rivo/global/historical/CNRM-ESM2-1
Number of downloaded files: 0


## Notes for the next processing steps

The downloaded files are stored in a generic project folder:

`../data/cmip6/downloads/{variable}/global/{experiment}/{model}/`

For example:

`../data/cmip6/downloads/rivo/global/ssp370/CNRM-ESM2-1/`

This folder can be used as input for later preprocessing, spatial subsetting, temporal aggregation, or basin statistics.
